In [ ]:
import pandas as pb
import numpy as np

from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

from datasets import load_dataset
from huggingface_hub import hf_hub_download

import utils

In [ ]:
onehot_events_df = utils.read_and_onehot_events("../data-processing/events.csv")

In [ ]:
# Load dataset splits
train_dataset = load_dataset("sookiemonster/asrs-narratives", split="train")
validation_dataset = load_dataset("sookiemonster/asrs-narratives", split="validation")
test_dataset = load_dataset("sookiemonster/asrs-narratives", split="test")

train_acns = set(train_dataset["acn"])
validation_acns = set(validation_dataset["acn"])
test_acns = set(test_dataset["acn"])

print(len(train_acns))
print(len(validation_acns))
print(len(test_acns))

In [ ]:
# Split up the main dataframe
train_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(train_acns)]
validation_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(validation_acns)]
test_events_df = onehot_events_df.iloc[onehot_events_df.index.isin(test_acns)]

print(len(train_events_df))
print(len(validation_events_df))
print(len(test_events_df))

One vs. Rest Classification (With SVMs) - Unbalanced

In [ ]:
svc = SVC()
ovr = OneVsRestClassifier(svc)

In [ ]:
trainX = train_events_df.drop(columns="assessments_primary_problem")
trainY = train_events_df["assessments_primary_problem"]

testX = test_events_df.drop(columns="assessments_primary_problem")
testY = test_events_df["assessments_primary_problem"]

In [ ]:
ovr.fit(trainX, trainY)

In [ ]:
ovr.score(testX, testY)

In [ ]:
predY = ovr.predict(testX)
utils.visualize_eval(testY, predY, "OVR Unbalanced")

One vs. Rest with SMOTE

In [ ]:
svc_bal = SVC()#class_weight="balanced")
ovr_bal = OneVsRestClassifier(svc_bal)
smt = SMOTE(k_neighbors=1)

In [ ]:
balanced_trainX, balanced_trainY = smt.fit_resample(trainX, trainY)

In [ ]:
ovr_bal.fit(balanced_trainX, balanced_trainY)

In [ ]:
ovr_bal.score(testX, testY)

In [ ]:
balanced_predY = ovr_bal.predict(testX)
utils.visualize_eval(testY, balanced_predY, "OVR Balanced")

SMOTE (Using more neighbors)

In [ ]:
svc_bal_k10 = SVC()#class_weight="balanced")
ovr_bal_k10 = OneVsRestClassifier(svc_bal)
smt_k10 = SMOTE(k_neighbors=5)

In [ ]:
train_events_noLB_df = train_events_df[train_events_df["assessments_primary_problem"] != "logbook_entry"]
validation_events_noLB_df = validation_events_df[validation_events_df["assessments_primary_problem"] != "logbook_entry"]
test_events_noLB_df = test_events_df[test_events_df["assessments_primary_problem"] != "logbook_entry"]

In [ ]:
# Remove logbook_entry primary problems to allow use of more neighbors for sample generation
trainX_noLB = train_events_noLB_df.drop(columns="assessments_primary_problem")
trainY_noLB = train_events_noLB_df["assessments_primary_problem"]

testX_noLB = test_events_noLB_df.drop(columns="assessments_primary_problem")
testY_noLB = test_events_noLB_df["assessments_primary_problem"]

trainX_noLB_bal, trainY_noLB_bal = smt_k10.fit_resample(trainX_noLB, trainY_noLB)

In [ ]:
ovr_bal_k10.fit(trainX_noLB_bal, trainY_noLB_bal)

In [ ]:
ovr_bal_k10.score(testX_noLB, testY_noLB)

In [ ]:
predY_noLB = ovr_bal_k10.predict(testX_noLB)
utils.visualize_eval(testY_noLB, predY_noLB, "OVR No Logbook Balanced")